In [ ]:
# Colab setup: clone data assets and enable interactive widgets
import os

try:
    import google.colab
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab] Cloning repository assets...")
    # Clean up any leftover temporary directory before cloning
    !rm -rf _repo_tmp
    !git clone https://github.com/mattjunior039/SelfDrivingCar.git _repo_tmp
    !cp -r _repo_tmp/data . 2>/dev/null || true
    !rm -rf _repo_tmp
    
    # CRITICAL: Enable ipywidgets for the interactive tuning sliders
    output.enable_custom_widget_manager()
    
    print("[Colab] Ready and interactive widgets enabled.")

# Phase 2 — Feature Learning with CNNs
### Self-Driving Car Curriculum · Student Workbook

---

## Where we left off

In Phase 1 you built a stop sign detector out of hand-written rules: six HSV thresholds, a morphology kernel size, an epsilon factor, an area cutoff. It worked beautifully on the photo you tuned it on, and then fell apart on shadows, sunsets, and red cars.

The core problem was this: **you had to personally think of every situation in advance.** Every number in that pipeline encoded one engineer's guess about what the world looks like. The world has more situations than you have patience.

## The idea behind this phase

What if, instead of *choosing* the filters, we let the computer **discover them from examples**?

That is the entire premise of a Convolutional Neural Network. A CNN is built from the same basic operation you already know — sliding a small kernel across an image — except the numbers inside the kernel are not typed in by you. They start as random noise and get nudged, thousands of times, until they become whatever filters happen to be most useful for the task.

```
  PHASE 1 (heuristic)                  PHASE 2 (learned)
  ───────────────────                  ─────────────────
  human picks HSV bounds               network picks kernel weights
  human picks kernel shape             human picks architecture
  human picks area threshold           human picks learning rate
  ~10 hand-tuned numbers               ~500,000 learned numbers
  works where you anticipated          works where your data covered
```

Notice the last line carefully. We are not escaping the problem of limited coverage — we are **trading hand-tuning for data collection**. That trade is usually worth it, but it is a trade, and you should be able to name what you gave up.

## The task

You will classify **64×64 pixel image patches** into three classes:

| Class | Label | What it looks like |
|---|---|---|
| `empty_road` | 0 | asphalt, lane markings, nothing in the way |
| `car` | 1 | a vehicle filling most of the patch |
| `pedestrian` | 2 | a person filling most of the patch |

A patch is a small crop, not a whole scene. **Hold onto that detail** — Section 4 is going to make a very big deal out of it.

## Learning objectives

By the end of this notebook you should be able to:
1. Explain a convolution as a sliding dot product, and predict what a given 3×3 kernel will do to an image.
2. Explain why learned kernels beat hand-designed ones, and what that costs you.
3. Build a small CNN in PyTorch and correctly compute the flattened dimension going into the first `Linear` layer.
4. Describe the roles of `ReLU`, `MaxPool2d`, and the loss function.
5. Read training/validation loss curves and identify overfitting.
6. Read a confusion matrix and say *which* mistakes the model makes, not just how many.
7. Articulate the **spatial bottleneck**: why a classifier cannot tell a car *where* an object is.

## Safety note

As in Phase 1: this is a toy trained on a tiny dataset. A real pedestrian-detection system is safety-critical, validated over millions of miles, and backed by redundant sensors. A model that is 95% accurate on a small test set is nowhere near good enough to be trusted with a human life. We will return to this.

---

## Setup

Run this once. Uncomment the install line if anything is missing.

> **GPU note:** this notebook is small enough to train on a CPU in a few minutes. If you have a GPU (CUDA) or an Apple Silicon Mac (MPS), the code below will pick it up automatically.

In [ ]:
# %pip install torch torchvision opencv-python numpy matplotlib scikit-learn ipywidgets

import os
import glob
import random
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["image.interpolation"] = "nearest"

# Reproducibility: same seed -> same random numbers -> same results.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Training device:", DEVICE)

---
# 1. Convolutions as Learnable Filters

## What a convolution actually does

A convolution slides a small grid of numbers — the **kernel** — across an image. At every position it multiplies each kernel number by the pixel underneath it, adds all the products together, and writes that single sum into the output.

```
   image patch        kernel          output pixel
  ┌────┬────┬────┐  ┌────┬────┬────┐
  │ 10 │ 10 │ 80 │  │ -1 │  0 │  1 │
  ├────┼────┼────┤  ├────┼────┼────┤   (10·-1)+(10·0)+(80·1)
  │ 12 │ 11 │ 82 │  │ -2 │  0 │  2 │ + (12·-2)+(11·0)+(82·2)  =  210
  ├────┼────┼────┤  ├────┼────┼────┤ + (11·-1)+(10·0)+(79·1)
  │ 11 │ 10 │ 79 │  │ -1 │  0 │  1 │
  └────┴────┴────┘  └────┴────┴────┘
```

Formally, for kernel $K$ of size $(2a{+}1) \times (2b{+}1)$:

$$\text{output}[i, j] = \sum_{m=-a}^{a} \sum_{n=-b}^{b} K[m, n] \cdot \text{image}[i + m,\; j + n]$$

In that example the left column was dark (~10) and the right column bright (~80). The kernel produced a large output, **210**, because it is designed to respond strongly to a vertical dark→bright transition. Run it over a flat region where everything is the same value and the positives and negatives cancel to roughly zero.

**That is the whole trick.** A kernel is a *pattern detector*. It outputs a big number where the image locally resembles the pattern, and near-zero everywhere else.

## Two properties worth naming

- **Locality.** Each output pixel depends only on a tiny neighborhood. Edges, corners, and textures are local things, so this is a sensible constraint.
- **Weight sharing.** The *same* nine numbers are used at every position in the image. A vertical-edge detector works in the top-left corner and the bottom-right corner alike. This is why CNNs need far less data than a network that treats every pixel independently — and why a feature learned on one part of an image transfers to every other part.

## The bridge to learning

The demo below uses kernels that humans designed decades ago — Sobel, Laplacian, box blur. They are clever, hand-crafted, and they work.

But here is the thing: **a `nn.Conv2d` layer is exactly this operation, with the nine numbers left blank.** Training fills them in. In Section 2 you will build a layer with 16 kernels; all 16 × 9 = 144 numbers start as random noise, and gradient descent gradually sculpts them into whatever detectors help separate cars from pedestrians from asphalt.

The remarkable empirical finding is that the first layer of a trained CNN almost always rediscovers edge and color-blob detectors that look strikingly like the hand-designed ones below. Nobody told it to. They are simply the most useful thing to compute first.

### Exercise 1.1 — Get a test image

Point `DEMO_IMAGE_PATH` at any photo you like (a road scene from Phase 1 is ideal). If the file is missing, the cell falls back to a synthetic test pattern with hard edges, soft gradients, and fine texture — which is actually quite useful for seeing what each kernel responds to.

In [ ]:
DEMO_IMAGE_PATH = "data/demo.jpg"


def make_test_pattern(size=256):
    """Synthetic fallback: hard edges, a gradient, a circle, and fine stripes."""
    img = np.full((size, size), 40, dtype=np.uint8)
    img[:, size // 2 :] = 200                                  # vertical edge
    img[: size // 3, :] = np.linspace(0, 255, size).astype(np.uint8)  # gradient
    cv2.circle(img, (size // 4, 2 * size // 3), size // 8, 255, -1)   # curved edges
    img[-size // 4 :, ::4] = 255                               # high-frequency stripes
    return img


_bgr = cv2.imread(DEMO_IMAGE_PATH, cv2.IMREAD_COLOR)
if _bgr is None:
    print(f"'{DEMO_IMAGE_PATH}' not found — using the synthetic test pattern.")
    demo_gray = make_test_pattern()
    demo_rgb = cv2.cvtColor(demo_gray, cv2.COLOR_GRAY2RGB)
else:
    if _bgr.shape[1] > 512:
        s = 512 / _bgr.shape[1]
        _bgr = cv2.resize(_bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    demo_rgb = cv2.cvtColor(_bgr, cv2.COLOR_BGR2RGB)
    demo_gray = cv2.cvtColor(_bgr, cv2.COLOR_BGR2GRAY)

print("Demo image shape:", demo_gray.shape)
plt.figure(figsize=(5, 5))
plt.imshow(demo_gray, cmap="gray", vmin=0, vmax=255)
plt.title("Grayscale input for the kernel demo")
plt.axis("off")
plt.show()

### Exercise 1.2 — The kernel library

Each kernel below is a 3×3 pattern detector. Read the comments and try to predict what each will do *before* you run the demo.

**The sum rule:** if a kernel's numbers sum to **1**, it preserves overall brightness (blur, sharpen). If they sum to **0**, flat regions become zero (black) and only *changes* survive — that is an edge detector. Check each one below against this rule.

Two kernels are left for you to fill in.

In [ ]:
KERNELS = {
    # Does nothing. The sanity check: output should equal input exactly.
    "identity": np.array([
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0],
    ], dtype=np.float32),

    # Box blur: plain average of the 9 neighbours. Sums to 1.
    "box blur": np.ones((3, 3), dtype=np.float32) / 9.0,

    # Gaussian blur: weighted average, centre counts most. Softer than box blur.
    "gaussian blur": np.array([
        [1, 2, 1],
        [2, 4, 2],
        [1, 2, 1],
    ], dtype=np.float32) / 16.0,

    # Sharpen: centre boosted, neighbours subtracted. Amplifies local contrast.
    "sharpen": np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0],
    ], dtype=np.float32),

    # Sobel X: left column negative, right column positive -> VERTICAL edges.
    "sobel x (vertical edges)": np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1],
    ], dtype=np.float32),

    # Laplacian: second derivative, fires on edges in ALL directions at once.
    "laplacian (all edges)": np.array([
        [ 0,  1,  0],
        [ 1, -4,  1],
        [ 0,  1,  0],
    ], dtype=np.float32),

    # Emboss: directional, gives a fake-3D lit-from-one-side look.
    "emboss": np.array([
        [-2, -1, 0],
        [-1,  1, 1],
        [ 0,  1, 2],
    ], dtype=np.float32),
}

# ------------------------------------------------------------------
# YOUR CODE HERE
#
# 1. "sobel y": the transpose of Sobel X. Top row negative, bottom row
#    positive, so it responds to HORIZONTAL edges instead of vertical.
#
# 2. "outline": centre = 8, all eight neighbours = -1. Sums to 0, so it
#    produces a strong outline on a black background.
# ------------------------------------------------------------------
KERNELS["sobel y (horizontal edges)"] = np.zeros((3, 3), dtype=np.float32)  # <-- replace
KERNELS["outline"] = np.zeros((3, 3), dtype=np.float32)                     # <-- replace


print(f"{'kernel':<28} {'sum':>7}   behaviour")
print("-" * 62)
for name, k in KERNELS.items():
    total = k.sum()
    kind = "preserves brightness" if abs(total - 1) < 1e-6 else (
        "edge detector (sums to 0)" if abs(total) < 1e-6 else "changes brightness"
    )
    print(f"{name:<28} {total:>7.2f}   {kind}")

### Exercise 1.3 — The interactive filter demo

Pick a kernel from the dropdown and watch what survives the filter.

**Things to try, in order:**
1. Start with **identity** — confirm the output matches the input. Now you trust the machinery.
2. Switch to **sobel x**. Which edges light up? Now **sobel y**. Compare directly: the same image, two kernels, two completely different "opinions" about what matters.
3. Try **laplacian** and turn on `absolute` — negative responses get flipped positive, so both dark→bright and bright→dark edges show up.
4. Crank **apply_times** to 3 with **box blur**. Repeated blurring destroys fine detail permanently. This is what information loss looks like.
5. Try **sharpen** on an already-blurry image, then on a sharp one. Notice it amplifies noise as eagerly as it amplifies real detail.

> **Why `float32` and `absolute`?** Edge kernels produce *negative* numbers, and a `uint8` image cannot store those — they would clip to 0 and you would silently lose half the signal. So we filter in floating point, optionally take the absolute value, and only convert back for display.

In [ ]:
def apply_kernel(gray, kernel, times=1, absolute=False):
    """Apply `kernel` to a grayscale image with cv2.filter2D, in float32."""
    out = gray.astype(np.float32)

    for _ in range(times):
        # ------------------------------------------------------------------
        # YOUR CODE HERE
        # Apply the convolution with cv2.filter2D.
        #   cv2.filter2D(src, ddepth, kernel)
        # Use ddepth=-1 to keep the same depth as `src` (which is float32 here).
        # ------------------------------------------------------------------
        out = out  # <-- replace with the cv2.filter2D call

    if absolute:
        out = np.abs(out)

    return out


def show_filter(kernel_name, apply_times, absolute, normalize_display):
    kernel = KERNELS[kernel_name]
    filtered = apply_kernel(demo_gray, kernel, times=apply_times, absolute=absolute)

    if normalize_display:
        lo, hi = filtered.min(), filtered.max()
        display_img = (filtered - lo) / (hi - lo + 1e-8)
        vmin, vmax = 0.0, 1.0
    else:
        display_img = np.clip(filtered, 0, 255)
        vmin, vmax = 0, 255

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(demo_gray, cmap="gray", vmin=0, vmax=255)
    axes[0].set_title("Input")
    axes[0].axis("off")

    axes[1].imshow(display_img, cmap="gray", vmin=vmin, vmax=vmax)
    axes[1].set_title(f"'{kernel_name}'  ×{apply_times}")
    axes[1].axis("off")

    # Render the 3x3 kernel itself as an annotated heatmap.
    axes[2].imshow(kernel, cmap="RdBu_r", vmin=-np.abs(kernel).max() - 1e-6,
                   vmax=np.abs(kernel).max() + 1e-6)
    for r in range(3):
        for c in range(3):
            axes[2].text(c, r, f"{kernel[r, c]:.2f}", ha="center", va="center",
                         fontsize=13, fontweight="bold")
    axes[2].set_title(f"the kernel (sum = {kernel.sum():.2f})")
    axes[2].set_xticks([])
    axes[2].set_yticks([])

    plt.tight_layout()
    plt.show()

    print(f"Output range: [{filtered.min():.1f}, {filtered.max():.1f}]")


demo = widgets.interactive(
    show_filter,
    kernel_name=widgets.Dropdown(
        options=list(KERNELS.keys()), value="identity", description="kernel:"
    ),
    apply_times=widgets.IntSlider(value=1, min=1, max=5, description="apply ×"),
    absolute=widgets.Checkbox(value=False, description="absolute value"),
    normalize_display=widgets.Checkbox(value=True, description="normalize display"),
)
display(demo)

### Exercise 1.4 — A convolution layer, before training

Here is the punchline of Section 1. The cell below creates a real PyTorch `nn.Conv2d` layer with 8 random kernels and runs your demo image through it.

The outputs are **garbage** — random kernels detect random things. Hold that image in your head. In Section 3 these same weights will be shaped by data into something useful, and the difference between these two pictures *is* machine learning.

In [ ]:
random_conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

print("Weight tensor shape:", tuple(random_conv.weight.shape),
      "  (out_channels, in_channels, kH, kW)")
print("Learnable numbers in this one layer:",
      sum(p.numel() for p in random_conv.parameters()))

# (1, 1, H, W): batch of 1, one channel, scaled to roughly 0-1.
x = torch.from_numpy(demo_gray.astype(np.float32) / 255.0)[None, None]

with torch.no_grad():
    feature_maps = random_conv(x)

print("Output shape:", tuple(feature_maps.shape), " <- 8 feature maps, same H and W\n")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, i].numpy(), cmap="gray")
    ax.set_title(f"random kernel #{i}", fontsize=9)
    ax.axis("off")
fig.suptitle("8 UNTRAINED kernels — this is what random weights see", fontsize=13)
plt.tight_layout()
plt.show()

> ### ✍️ Reflection 1
>
> 1. The Sobel X kernel sums to zero. Explain, using the sliding dot product, why that makes a flat gray wall come out black.
> 2. Sobel X and Sobel Y are the *same nine numbers rotated*. Why does a network need both? What would it miss with only one?
> 3. A `Conv2d(1, 8, 3)` layer has 80 learnable numbers (8 kernels × 9 weights + 8 biases). A `Linear` layer connecting every pixel of a 64×64 image to 8 outputs would need over 32,000. Both look at the whole image. Why is the convolution so much cheaper, and what assumption about images makes that cheapness *reasonable* rather than reckless?

*Your answers:*

1. 
2. 
3. 

---
# 2. Building the Architecture

## The data first

Before the model, we need examples. The expected folder layout is:

```
data/patches/
├── empty_road/   img_0001.jpg, img_0002.jpg, ...
├── car/          img_0001.jpg, ...
└── pedestrian/   img_0001.jpg, ...
```

Aim for **at least 100 images per class** — more is much better. You can crop these by hand from dashcam footage, or use a public dataset.

If those folders are empty, the next cell generates a **synthetic stand-in dataset** so the notebook still runs end to end. The synthetic data is deliberately simplistic (colored blobs on textured asphalt). It will train to very high accuracy, and that high number is *not* a sign your model is good — it is a sign the task was too easy. Use real patches when you can.

## Anatomy of a CNN

A small classifier has two halves:

```
 ─────────── FEATURE EXTRACTOR ───────────   ──── CLASSIFIER HEAD ────

 input        block 1           block 2       flatten      dense
 3×64×64  →  16×32×32     →    32×16×16   →   8192    →   3 scores
             conv+relu+pool    conv+relu+pool

             "what patterns      "what patterns   "combine everything
              are present?"       of patterns?"    into a decision"
```

### The three ingredients

**`Conv2d`** — the learnable filters from Section 1. `padding=1` with `kernel_size=3` keeps height and width unchanged, which makes the arithmetic easy to track.

**`ReLU`** — $\text{ReLU}(x) = \max(0, x)$. Clamps negatives to zero. Without it, stacking layers is pointless: a chain of linear operations collapses into one linear operation, so a 50-layer network would have exactly the same expressive power as a 1-layer one. ReLU is the non-linearity that makes depth *mean* something.

**`MaxPool2d(2)`** — takes the strongest response in each 2×2 block, halving height and width. Two benefits: it cuts computation 4×, and it grants a little **translation tolerance** — shift the input a pixel or two and the max often doesn't change. Remember this word *tolerance*; Section 4 shows how it becomes a liability.

## The dimension-tracking rule

The single most common beginner error in PyTorch is a shape mismatch at the first `Linear` layer. Learn this rule:

| Layer | Effect on shape |
|---|---|
| `Conv2d(in, out, 3, padding=1)` | channels `in`→`out`; H, W unchanged |
| `ReLU()` | nothing changes |
| `MaxPool2d(2)` | channels unchanged; H, W each ÷ 2 |
| `flatten` | `(C, H, W)` → one vector of length `C × H × W` |

So for a 3×64×64 input:

$$3{\times}64{\times}64 \xrightarrow{\text{block 1}} 16{\times}32{\times}32 \xrightarrow{\text{block 2}} 32{\times}16{\times}16 \xrightarrow{\text{flatten}} 32 \cdot 16 \cdot 16 = 8192$$

You will need that 8192 shortly. Derive it yourself rather than copying it — if you change the channel counts, it changes.

In [ ]:
CLASSES = ["empty_road", "car", "pedestrian"]
PATCH_SIZE = 64
PATCH_DIR = os.path.join("data", "patches")

for c in CLASSES:
    os.makedirs(os.path.join(PATCH_DIR, c), exist_ok=True)


def _synthesize(cls, rng):
    """Crude stand-in patch so the notebook runs without real data."""
    base = rng.integers(70, 110)
    img = np.full((PATCH_SIZE, PATCH_SIZE, 3), base, dtype=np.uint8)
    img = cv2.add(img, rng.integers(0, 28, img.shape, dtype=np.uint8))  # asphalt grain

    if cls == "empty_road":
        if rng.random() < 0.5:  # lane marking
            x = rng.integers(8, PATCH_SIZE - 12)
            img[:, x : x + rng.integers(3, 7)] = rng.integers(180, 245)
    elif cls == "car":
        color = tuple(int(v) for v in rng.integers(40, 240, 3))
        x, y = rng.integers(4, 16), rng.integers(16, 30)
        w, h = rng.integers(34, 52), rng.integers(20, 30)
        cv2.rectangle(img, (x, y), (min(x + w, 63), min(y + h, 63)), color, -1)
        cv2.rectangle(img, (x + 6, y + 3), (min(x + w - 6, 63), y + 11), (30, 40, 60), -1)
        for wx in (x + 9, x + w - 9):  # wheels
            cv2.circle(img, (min(wx, 63), min(y + h, 63)), 4, (20, 20, 20), -1)
    else:  # pedestrian: tall and narrow
        color = tuple(int(v) for v in rng.integers(30, 220, 3))
        cx, top = rng.integers(22, 42), rng.integers(6, 14)
        cv2.circle(img, (cx, top + 6), 5, (200, 170, 140), -1)             # head
        cv2.rectangle(img, (cx - 7, top + 12), (cx + 7, top + 34), color, -1)  # torso
        cv2.line(img, (cx - 4, top + 34), (cx - 6, min(top + 50, 63)), color, 3)
        cv2.line(img, (cx + 4, top + 34), (cx + 6, min(top + 50, 63)), color, 3)

    if rng.random() < 0.4:  # random lighting shift
        img = np.clip(img.astype(np.int16) + rng.integers(-45, 45), 0, 255).astype(np.uint8)
    return img


def ensure_dataset(n_per_class=250):
    rng = np.random.default_rng(SEED)
    made = 0
    for cls in CLASSES:
        folder = os.path.join(PATCH_DIR, cls)
        existing = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".png"))]
        if len(existing) >= 20:
            continue
        for i in range(n_per_class):
            cv2.imwrite(os.path.join(folder, f"synth_{i:04d}.png"), _synthesize(cls, rng))
            made += 1
    return made


n_made = ensure_dataset()
IS_SYNTHETIC = n_made > 0
if IS_SYNTHETIC:
    print(f"No real data found — generated {n_made} synthetic patches.")
    print("WARNING: synthetic accuracy will be unrealistically high. Use real crops when you can.\n")

for cls in CLASSES:
    n = len(glob.glob(os.path.join(PATCH_DIR, cls, "*")))
    print(f"  {cls:<12} {n:>5} images")

In [ ]:
class RoadPatchDataset(Dataset):
    """Loads 64x64 RGB patches from data/patches/<class_name>/."""

    def __init__(self, root=PATCH_DIR, classes=CLASSES, augment=False):
        self.classes = classes
        self.augment = augment
        self.samples = []
        for label, cls in enumerate(classes):
            for path in sorted(glob.glob(os.path.join(root, cls, "*"))):
                if path.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    self.samples.append((path, label))
        if not self.samples:
            raise RuntimeError(f"No images found under '{root}'.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        bgr = cv2.resize(bgr, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        if self.augment and random.random() < 0.5:
            rgb = np.fliplr(rgb).copy()  # a mirrored car is still a car

        # HWC uint8 [0,255]  ->  CHW float32 [0,1], which is what Conv2d expects.
        tensor = torch.from_numpy(rgb.astype(np.float32) / 255.0).permute(2, 0, 1)
        return tensor, label


full_dataset = RoadPatchDataset(augment=False)
print(f"Total patches: {len(full_dataset)}")

counts = np.bincount([lbl for _, lbl in full_dataset.samples], minlength=len(CLASSES))
for name, n in zip(CLASSES, counts):
    print(f"  {name:<12} {n:>5}")
if counts.max() > 3 * max(counts.min(), 1):
    print("\nNOTE: your classes are imbalanced. The model may learn to just guess the")
    print("common class. Watch for this in the confusion matrix in Section 4.")

### Exercise 2.1 — Look at your data before you model it

Never train on data you have not looked at. Mislabeled files, all-black images, and duplicated crops are extremely common and will quietly wreck your results.

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 8, figsize=(14, 5.5))
for row, cls in enumerate(CLASSES):
    idxs = [i for i, (_, lbl) in enumerate(full_dataset.samples) if lbl == row]
    for col, ax in enumerate(axes[row]):
        ax.axis("off")
        if col < len(idxs):
            img, _ = full_dataset[random.choice(idxs)]
            ax.imshow(img.permute(1, 2, 0).numpy())
    axes[row, 0].set_ylabel(cls)
    axes[row, 0].axis("on")
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])
plt.suptitle("Random samples from each class", fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 2.2 — Build `MiniRoadCNN`

Block 1 is written for you as a worked example. **Copy its structure** for block 2, then build the classifier head.

**Hints:**
- Block 2 should take 16 channels in and produce 32 out, with the same `kernel_size=3, padding=1`.
- For the head, work out the flattened size using the table above. Do the arithmetic on paper.
- `Dropout` randomly zeroes some activations during training. It forces the network not to depend on any single feature, which fights overfitting. It automatically turns itself off during evaluation.
- The final layer outputs **3 raw scores (logits)** — one per class. Do **not** put a softmax here; `CrossEntropyLoss` applies it internally, and doing it twice silently hurts training.

In [ ]:
class MiniRoadCNN(nn.Module):
    """A small CNN: 3x64x64 patch in, 3 class logits out."""

    def __init__(self, n_classes=len(CLASSES), dropout=0.3):
        super().__init__()

        # ---------------- BLOCK 1 — fully written for you ----------------
        # 3x64x64 -> conv(16) -> 16x64x64 -> relu -> pool -> 16x32x32
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        # ---------------- BLOCK 2 ----------------
        # ==============================================================
        # YOUR CODE HERE
        #
        # Build a second block with the SAME three-layer pattern:
        #     Conv2d(16 -> 32, kernel_size=3, padding=1)
        #     ReLU()
        #     MaxPool2d(2)
        #
        # It should take 16x32x32 and produce 32x16x16.
        # ==============================================================
        self.block2 = nn.Identity()  # <-- replace with nn.Sequential(...)

        # ---------------- CLASSIFIER HEAD ----------------
        # ==============================================================
        # YOUR CODE HERE
        #
        # 1. Work out `flat_features`: after block 2 the tensor is
        #    (32 channels, 16 height, 16 width). Multiply them.
        #
        # 2. Build the head as nn.Sequential:
        #        nn.Flatten()
        #        nn.Linear(flat_features, 64)
        #        nn.ReLU()
        #        nn.Dropout(dropout)
        #        nn.Linear(64, n_classes)
        #
        # Remember: NO softmax at the end.
        # ==============================================================
        flat_features = None  # <-- replace with your computed number
        self.head = nn.Identity()  # <-- replace with nn.Sequential(...)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.head(x)
        return x

### Exercise 2.3 — Shape check

**Always** run a fake batch through a new model before training it. Thirty seconds here saves half an hour of confusing errors later.

This cell prints the shape after every stage, so if something is wrong you will see exactly which block did it.

In [ ]:
model = MiniRoadCNN()
dummy = torch.randn(4, 3, PATCH_SIZE, PATCH_SIZE)  # batch of 4 fake patches

with torch.no_grad():
    a = model.block1(dummy)
    b = model.block2(a)
    out = model.head(b)

print(f"input        {tuple(dummy.shape)}")
print(f"after block1 {tuple(a.shape)}   expected (4, 16, 32, 32)")
print(f"after block2 {tuple(b.shape)}   expected (4, 32, 16, 16)")
print(f"after head   {tuple(out.shape)}   expected (4, 3)")

assert a.shape == (4, 16, 32, 32), "Block 1 output is wrong."
assert b.shape == (4, 32, 16, 16), "Block 2 is not built correctly yet."
assert out.shape == (4, 3), "The head should output one score per class."

total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {total:,}")
print("\nWhere they live:")
for name, module in [("block1", model.block1), ("block2", model.block2), ("head", model.head)]:
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<8} {n:>10,}  ({100 * n / max(total, 1):5.1f}%)")

print("\nAll shape checks passed.")

> ### ✍️ Reflection 2
>
> 1. Look at the parameter breakdown. The convolutional blocks do most of the *work*, but one part holds the overwhelming majority of the *parameters*. Which one, and why does flattening create so many weights?
> 2. What would happen if you removed every `ReLU`? (Be precise: say something about what the whole network reduces to.)
> 3. Suppose you fed in a 128×128 patch instead of 64×64. Which single layer would crash, and why? What does that tell you about how rigidly a classifier is tied to its input size?

*Your answers:*

1. 
2. 
3. 

---
# 3. The Optimization Loop

## How learning actually happens

Training is a loop of four steps, repeated for every batch of images:

```
  ┌─────────────────────────────────────────────────────────┐
  │  1. FORWARD    push images through -> get logits         │
  │  2. LOSS       compare logits to true labels -> a number │
  │  3. BACKWARD   compute how each weight affected the loss │
  │  4. STEP       nudge every weight to reduce the loss     │
  └─────────────────────────────────────────────────────────┘
                        repeat ~thousands of times
```

### `CrossEntropyLoss`

Converts the 3 raw logits into probabilities (softmax), then measures how much probability the model gave to the *correct* answer:

$$\mathcal{L} = -\log(p_{\text{correct}})$$

Confidently right ($p = 0.99$) → loss $\approx 0.01$. Unsure ($p = 0.33$) → loss $\approx 1.10$. Confidently **wrong** ($p = 0.01$) → loss $\approx 4.6$. The $-\log$ punishes confident mistakes enormously, which is exactly the behaviour you want from something that might one day look at pedestrians.

Useful sanity check: with 3 balanced classes, an untrained model should start near $-\log(1/3) \approx 1.10$. **If your loss does not start around 1.1, something is wrong before you even begin.**

### `Adam`

The optimizer decides *how* to nudge each weight. Adam keeps a running estimate of each weight's recent gradients and adapts the step size per weight — big steps for weights that consistently want to move, small cautious steps for erratic ones. It is the sensible default. `lr=1e-3` is the standard starting point.

### `zero_grad()` — the classic bug

PyTorch **accumulates** gradients by default. If you forget `optimizer.zero_grad()`, each batch's gradients pile on top of the last, your effective step size balloons, and training diverges into NaN. It fails loudly, but the cause is not obvious the first time.

## Train / validation split

We hold back 20% of the data that the model **never trains on**. Why? Because a model with 500,000 parameters can simply *memorize* a few hundred images. Accuracy on data it memorized tells you nothing.

Read the two curves together:

| Pattern | Name | What to do |
|---|---|---|
| both falling | healthy learning | keep going |
| train ↓, val ↑ | **overfitting** — memorizing | more data, more dropout, stop earlier |
| both flat and high | **underfitting** | bigger model, higher `lr`, train longer |
| loss = NaN | diverged | lower `lr`, check `zero_grad()` |

The gap between the curves is the single most informative thing on the plot.

In [ ]:
BATCH_SIZE = 32
VAL_FRACTION = 0.2

n_val = int(len(full_dataset) * VAL_FRACTION)
n_train = len(full_dataset) - n_val

train_subset, val_subset = random_split(
    full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)

# Augmentation belongs on training data only — never on validation.
train_subset.dataset = RoadPatchDataset(augment=True)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training patches  : {n_train}")
print(f"Validation patches: {n_val}")
print(f"Batches per epoch : {len(train_loader)}")

### Exercise 3.1 — The training script

This cell is **complete** — you do not need to edit it to make it run. Read it carefully instead; the four numbered steps in `train_one_epoch` are the heart of all deep learning.

Note `model.train()` vs `model.eval()`. These toggle Dropout on and off. Forgetting `model.eval()` before validating is a very common bug and makes your validation numbers randomly worse than they should be. The `torch.no_grad()` block likewise tells PyTorch not to track gradients during evaluation, which saves memory and time.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()  # Dropout ON
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()             # 0. clear old gradients
        logits = model(images)            # 1. forward
        loss = criterion(logits, labels)  # 2. loss
        loss.backward()                   # 3. backward
        optimizer.step()                  # 4. update weights

        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()  # Dropout OFF
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def plot_live(history, epoch, n_epochs):
    """Redraw the loss and accuracy curves in place after each epoch."""
    clear_output(wait=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], "o-", label="train")
    axes[0].plot(epochs, history["val_loss"], "s-", label="validation")
    axes[0].axhline(np.log(len(CLASSES)), ls="--", c="gray",
                    label=f"random guessing ({np.log(len(CLASSES)):.2f})")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("cross-entropy loss")
    axes[0].set_title(f"Loss — epoch {epoch}/{n_epochs}")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], "o-", label="train")
    axes[1].plot(epochs, history["val_acc"], "s-", label="validation")
    axes[1].axhline(1 / len(CLASSES), ls="--", c="gray", label="random guessing")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")
    axes[1].set_ylim(0, 1.03)
    axes[1].set_title("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    gap = history["val_loss"][-1] - history["train_loss"][-1]
    print(f"epoch {epoch:>3}/{n_epochs}  "
          f"train loss {history['train_loss'][-1]:.4f}  acc {history['train_acc'][-1]:.3f}   |   "
          f"val loss {history['val_loss'][-1]:.4f}  acc {history['val_acc'][-1]:.3f}   "
          f"(gap {gap:+.3f})")

### Exercise 3.2 — Train it

Run the cell and watch the curves redraw after every epoch.

**What to watch for:**
- Epoch 1 loss should be near **1.10**. Much higher means something is misconfigured.
- The best model (lowest validation loss) is saved to `best_model.pt` automatically, so a later overfitting epoch cannot destroy your good result.
- If the validation curve turns upward while the training curve keeps falling — that is overfitting, live on screen. Note the epoch where it happens.

In [ ]:
N_EPOCHS = 20
LEARNING_RATE = 1e-3

model = MiniRoadCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
best_epoch = 0
start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    va_loss, va_acc = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    if va_loss < best_val_loss:
        best_val_loss, best_epoch = va_loss, epoch
        torch.save(model.state_dict(), "best_model.pt")

    plot_live(history, epoch, N_EPOCHS)

print(f"\nFinished in {time.time() - start:.1f}s")
print(f"Best validation loss {best_val_loss:.4f} at epoch {best_epoch} (saved to best_model.pt)")

model.load_state_dict(torch.load("best_model.pt"))
print("Reloaded the best checkpoint.")

### Exercise 3.3 — What did the first layer learn?

Remember the random noise from Exercise 1.4? Those same 16 kernels have now been shaped by your data. Look for oriented edges, bright/dark contrasts, and color-opponent patterns.

> **Manage your expectations:** with a small dataset the kernels may still look noisy. Big, clean, textbook-pretty filters come from training on hundreds of thousands of images. Look for *structure*, not beauty.

In [ ]:
weights = model.block1[0].weight.detach().cpu().clone()  # (16, 3, 3, 3)

# Normalize each kernel to 0-1 independently so it is visible as a colour swatch.
w_min = weights.amin(dim=(1, 2, 3), keepdim=True)
w_max = weights.amax(dim=(1, 2, 3), keepdim=True)
normalized = (weights - w_min) / (w_max - w_min + 1e-8)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(normalized[i].permute(1, 2, 0).numpy())
    ax.set_title(f"#{i}", fontsize=8)
    ax.axis("off")
fig.suptitle("16 LEARNED kernels from block 1 — compare with the random ones earlier", fontsize=12)
plt.tight_layout()
plt.show()

# And the feature maps a real patch produces.
sample_img, sample_label = full_dataset[random.randrange(len(full_dataset))]
with torch.no_grad():
    maps = model.block1(sample_img[None].to(DEVICE)).cpu()[0]

fig, axes = plt.subplots(2, 9, figsize=(16, 4))
axes[0, 0].imshow(sample_img.permute(1, 2, 0).numpy())
axes[0, 0].set_title(f"input\n({CLASSES[sample_label]})", fontsize=9)
axes[1, 0].axis("off")
for i, ax in enumerate(list(axes[0, 1:]) + list(axes[1, 1:])):
    ax.imshow(maps[i].numpy(), cmap="viridis")
    ax.set_title(f"map {i}", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("What each learned kernel responds to (bright = strong response)", fontsize=12)
plt.tight_layout()
plt.show()

### Exercise 3.4 — Break it on purpose

Change **one** hyperparameter, re-run the training cell, and record what happens. Do not skip this — reading about a diverging loss is nothing like watching one.

Suggested experiments:
- `LEARNING_RATE = 1.0` — far too large. Watch the loss explode or stick at 1.10.
- `LEARNING_RATE = 1e-6` — far too small. Watch it barely move.
- `N_EPOCHS = 100` — train far past the point of usefulness and find the overfitting elbow.
- `dropout=0.0` in `MiniRoadCNN` — does the train/val gap widen?
- Comment out `optimizer.zero_grad()` — see the accumulation bug for yourself, then put it back.

*Record your experiments:*

| What I changed | Value | Final train acc | Final val acc | What the curves did |
|---|---|---|---|---|
| *(baseline)* | — | | | |
| `LEARNING_RATE` | 1.0 | | | |
| `LEARNING_RATE` | 1e-6 | | | |
| `N_EPOCHS` | 100 | | | |
| `dropout` | 0.0 | | | |

---
# 4. Evaluation & The Spatial Bottleneck

## Accuracy is a bad summary

"94% accurate" hides everything that matters. Consider a dataset that is 90% empty road: a model that outputs `empty_road` for literally every input scores 90%. It is useless and dangerous, and the accuracy number looks great.

What you actually need to know is **which mistakes** it makes. Not all errors cost the same:

| Confusion | Consequence |
|---|---|
| `car` → `pedestrian` | wrong label, but the car still brakes for an obstacle. Mild. |
| `empty_road` → `car` | phantom braking. Bad, and a real cause of rear-end collisions. |
| `pedestrian` → `empty_road` | **the vehicle does not stop for a person.** Catastrophic. |

A confusion matrix shows all of this at once. Rows are the truth, columns are the prediction. A perfect model has everything on the diagonal; every off-diagonal cell is a specific, nameable failure.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

model.eval()
y_true, y_pred, y_conf = [], [], []

with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(DEVICE))
        probs = F.softmax(logits, dim=1)
        conf, preds = probs.max(dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_conf.extend(conf.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ConfusionMatrixDisplay(
    confusion_matrix(y_true, y_pred), display_labels=CLASSES
).plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d")
axes[0].set_title("Confusion matrix (counts)")

ConfusionMatrixDisplay(
    confusion_matrix(y_true, y_pred, normalize="true"), display_labels=CLASSES
).plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Normalized by true class (row-wise recall)")

for ax in axes:
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

ped = CLASSES.index("pedestrian")
empty = CLASSES.index("empty_road")
missed = int(confusion_matrix(y_true, y_pred)[ped, empty])
print(f"SAFETY-CRITICAL CELL: pedestrians called 'empty_road' = {missed}")
print("In a real vehicle, each one of those is a person the car did not see.")

### Exercise 4.1 — Look at the actual mistakes

Numbers tell you *that* it failed. Only the images tell you *why*. Pay attention to the confidence values: a model that is confidently wrong is far more worrying than one that is uncertain, because downstream systems have no signal that anything went awry.

In [ ]:
val_images = [val_subset[i][0] for i in range(len(val_subset))]
wrong = np.where(y_true != y_pred)[0]

print(f"{len(wrong)} mistakes out of {len(y_true)} validation patches.")

if len(wrong) == 0:
    print("No errors. On real-world data this is essentially never a good sign —")
    print("it usually means the task is too easy or the validation set is too small.")
else:
    order = wrong[np.argsort(-y_conf[wrong])]  # most confident mistakes first
    show = order[:12]
    n_cols = 6
    n_rows = int(np.ceil(len(show) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.3 * n_cols, 2.9 * n_rows))
    for ax, idx in zip(np.atleast_1d(axes).flat, show):
        ax.imshow(val_images[idx].permute(1, 2, 0).numpy())
        ax.set_title(
            f"true: {CLASSES[y_true[idx]]}\npred: {CLASSES[y_pred[idx]]} ({y_conf[idx]:.2f})",
            fontsize=8, color="darkred",
        )
        ax.axis("off")
    for ax in np.atleast_1d(axes).flat[len(show):]:
        ax.axis("off")
    fig.suptitle("Most CONFIDENT mistakes", fontsize=13)
    plt.tight_layout()
    plt.show()

### Exercise 4.2 — The bottleneck, demonstrated

Now the important part.

The cell below takes **one** patch containing an object and slides it to different positions inside a 64×64 frame — left, right, top, bottom, corner. The object is identical every time. Only its **location** changes.

Watch the predicted class and confidence. Then ask yourself the question that defines Phase 3: *given only these outputs, could you tell a car where to steer?*

In [ ]:
# Grab one non-empty patch to use as our movable "object".
obj_idx = next(i for i, (_, lbl) in enumerate(full_dataset.samples) if CLASSES[lbl] != "empty_road")
obj_tensor, obj_label = full_dataset[obj_idx]
obj_np = (obj_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

# Shrink it so there is room to move it around.
small = cv2.resize(obj_np, (32, 32), interpolation=cv2.INTER_AREA)

positions = {
    "top-left": (0, 0),
    "top-right": (0, 32),
    "centre": (16, 16),
    "bottom-left": (32, 0),
    "bottom-right": (32, 32),
}

fig, axes = plt.subplots(1, len(positions), figsize=(3 * len(positions), 3.8))

for ax, (label, (r, c)) in zip(axes, positions.items()):
    canvas = np.full((PATCH_SIZE, PATCH_SIZE, 3), 95, dtype=np.uint8)
    canvas[r : r + 32, c : c + 32] = small

    t = torch.from_numpy(canvas.astype(np.float32) / 255.0).permute(2, 0, 1)[None]
    with torch.no_grad():
        probs = F.softmax(model(t.to(DEVICE)), dim=1)[0].cpu().numpy()

    ax.imshow(canvas)
    ax.set_title(f"{label}\n{CLASSES[probs.argmax()]}  {probs.max():.2f}", fontsize=9)
    ax.axis("off")

fig.suptitle("Same object, five positions — the output is 3 numbers every time", fontsize=13)
plt.tight_layout()
plt.show()

print("The model's ENTIRE output for each image above is a vector of length 3.")
print("Nowhere in those 3 numbers is there room to encode an (x, y) coordinate.")

---
## The Spatial Bottleneck

### What the architecture threw away

Trace the shapes through your own network one more time:

```
  3 × 64 × 64   =  12,288 numbers   ← every pixel, fully located
 16 × 32 × 32   =  16,384 numbers   ← still a spatial grid
 32 × 16 × 16   =   8,192 numbers   ← still a spatial grid, coarser
        ↓ nn.Flatten()
      8,192     =   one long vector  ← the grid is now an arbitrary ordering
        ↓ nn.Linear(8192, 64)
         64     ←  everything mixed together
        ↓ nn.Linear(64, 3)
          3     ←  "car / pedestrian / empty", and NOTHING else
```

Up through block 2, position is still preserved — feature map cell $(4, 7)$ genuinely corresponds to a region in the upper-left of the image. **`nn.Flatten()` is where that dies.** It pours a 2-D grid into a 1-D list, and the very next `Linear` layer connects everything to everything, deliberately blending all positions together.

That blending is not a bug. It is precisely what makes a classifier robust: you *want* "car" to mean car whether it sits left or right in the patch. Every `MaxPool2d` throws away a little location detail on purpose, and flattening finishes the job.

But robustness to position and knowledge of position are **the same information**, and you cannot discard it and keep it.

### Why this is fatal for a vehicle

A real camera frame is 1920×1080, not 64×64, and it contains many objects at once. Your classifier, pointed at a full frame, can only say `"car"`. It cannot say:

- **Where?** Left lane or directly ahead? That is the difference between ignoring a vehicle and emergency braking.
- **How many?** One pedestrian or a crowd at a crosswalk?
- **How big / how far?** A 300-pixel-tall pedestrian is close. A 20-pixel one is far.
- **Moving which way?** Stepping into the road, or away from it?

Planning a trajectory requires coordinates. Your model does not produce coordinates. It cannot be patched into producing them — the information was destroyed by the architecture itself.

### The obvious fix, and why it fails

"Just slide the 64×64 classifier over the whole frame like a convolution kernel!" This is called the **sliding window** approach, and it genuinely was the state of the art for years. But:

- At stride 8 on a 1920×1080 frame, that is roughly 30,000 windows **per frame**.
- At 30 frames per second, that is ~900,000 forward passes per second.
- And objects come in many sizes, so you must repeat the whole sweep at several scales — call it 5× more.
- A car braking at 60 mph covers about 27 metres per second. You have *milliseconds*, not seconds.

Worse, nearby windows overlap heavily and recompute nearly identical convolutions over and over. It is enormously wasteful.

### What comes next

Phase 3 introduces **object detection** — architectures that keep the spatial grid all the way to the output and predict, for every cell in that grid, both *what* is there and *where* its box is. One forward pass, every object, with coordinates.

The key architectural move is simply: **don't flatten.** Everything else follows from that.

---

## ✍️ Final Reflection — Write-up

Answer in full sentences. This is the assessed portion of Phase 2.

**1. Learned vs. designed.** Compare the random kernels from Exercise 1.4 with the trained kernels from Exercise 3.3. What changed, and what *caused* it to change? Name one advantage learned filters have over the hand-picked Sobel/Laplacian kernels, and one thing they cost you that Phase 1's approach did not.

**2. Reading the curves.** Describe your own loss plot. Did the two curves separate? At roughly which epoch? Which row of the diagnosis table matches what you saw, and what would you change to improve it?

**3. The matrix.** Identify the largest off-diagonal cell in your confusion matrix. Explain, referring to the actual mistake images from Exercise 4.1, why the model plausibly confuses those two classes. Then say whether this error is *safe* or *dangerous* for a vehicle and defend your answer.

**4. The spatial bottleneck — the central question.** In your own words, explain why `MiniRoadCNN` cannot tell a self-driving car *where* an object is in a 1920×1080 video frame. Your answer must reference a specific layer in your architecture and explain exactly what information that layer destroys. Then explain why this is a consequence of the architecture and not merely a lack of training data — i.e. why training for a million more epochs would not fix it.

**5. Cost of the naive fix.** Estimate how many forward passes a sliding-window sweep would need for one 1920×1080 frame at stride 16 with a 64×64 window. (Roughly: $\lfloor (1920-64)/16 \rfloor \times \lfloor (1080-64)/16 \rfloor$.) Multiply by 30 FPS. Comment on whether this is plausible on hardware inside a car.

**6. Honest limitations.** Your validation accuracy is some number. List three concrete reasons that number *overstates* how well this model would perform on a real road tomorrow. (Consider: how the data was collected, what weather and times of day it covers, how big the validation set actually is, and — if you used the synthetic generator — what that data is missing.)

*Your write-up:*

**1.**

**2.**

**3.**

**4.**

**5.**

**6.**

---
## Stretch goals (optional)

- **Build a real sliding-window detector.** Loop your classifier over a full road photo at stride 16 and draw a box wherever confidence exceeds 0.9. Time it. Now you *feel* the cost from question 5 instead of just calculating it.
- **Add a third conv block.** 32 → 64 channels, giving 64×8×8. Recompute the flattened size. Does deeper actually help on your dataset, or just overfit faster?
- **Replace `Flatten` with `nn.AdaptiveAvgPool2d(1)`.** This makes the model accept *any* input size. Try feeding it 128×128. Why does it no longer crash? (This is a genuine hint about Phase 3.)
- **Stronger augmentation.** Add rotation, brightness jitter, and random crops. Does the train/val gap shrink?
- **Add a fourth class** — `cyclist` or `traffic_light`. How much new data do you need before it works? Compare that effort to adding a fourth class in Phase 1's heuristic pipeline.
- **Occlusion sensitivity.** Slide a gray square over an input patch and record how much the correct-class probability drops at each position. Plot it as a heatmap — you have just built a crude explanation of *where the model is looking*. Note that this heatmap has spatial information the model's output does not.

---

### ➡️ Next: Phase 3 — Object Detection
Keep the spatial grid. Predict boxes and classes in a single pass. Meet anchor boxes, IoU, and non-maximum suppression.